In [156]:
from tqdm import tqdm # Library untuk bar progrss
import time
import warnings # Library untuk handle warning
warnings.filterwarnings('ignore')
import json

# Library pemrosesan data
import pandas as pd
import numpy as np
import networkx as nx
from sklearn.neighbors import KDTree
from sklearn.neighbors import BallTree
from rtree import index
from shapely.geometry import LineString, Point, shape
from datetime import timedelta
from haversine import haversine
from pykalman import KalmanFilter
import datetime

# Library untuk visualisasi data
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import DBSCAN
from itertools import groupby
import math
import matplotlib.animation as animation
from geopy.distance import geodesic
from collections import Counter
import geopandas as gpd


# Library untuk metrik dan model selection
from sklearn.metrics import mean_squared_error as mse
from sklearn.model_selection import (
    KFold,
    cross_val_predict as cvp,
    cross_val_score as cvs
)

# Library-library untuk model prediksi
from sklearn import linear_model
from sklearn import ensemble
import xgboost as xgb
import lightgbm as lgbm
import catboost as cb

# Library untuk melakukan Hyperparameter Tuning
import optuna

# Library untuk menyimpan model
import pickle

directory = ''

file_path1 = 'jalanraya_ui_flowcoord.json'
file_path2 = 'routes.json'

# Open and read the JSON files
with open(file_path1, 'r') as f1, open(file_path2, 'r') as f2:
    road_data = json.load(f1)
    route_data = json.load(f2)

train = pd.read_csv(f'train.csv')
test = pd.read_csv(f'test.csv')

In [140]:
def process(train, test):

    # Convert the dictionary to a list of tuples (key, value) for RUTE_MERAH
    rute_merah_items = list(route_data["RUTE_MERAH"].items())
    
    # Find the index of "fisip_1" and insert the new coordinate after it
    for i, (key, value) in enumerate(rute_merah_items):
        if key == "fisip_1":
            rute_merah_items.insert(i + 1, ("fpsi_1", [-6.362195, 106.830708]))
            break
    
    # Find the index of "stasiun_ui_asrama_01" and insert "menwa_asrama_01" after it
    for i, (key, value) in enumerate(rute_merah_items):
        if key == "stasiun_ui_asrama_01":
            rute_merah_items.insert(i + 1, ("menwa_asrama_01", [-6.353454, 106.831660]))
            break
    
    # Convert back to a dictionary
    route_data["RUTE_MERAH"] = dict(rute_merah_items)
    
    # Convert the dictionary to a list of tuples (key, value) for RUTE_BIRU
    rute_biru_items = list(route_data["RUTE_BIRU"].items())
    
    # Find the index of "stasiun_ui_asrama_01" and insert "menwa_asrama_01" after it
    for i, (key, value) in enumerate(rute_biru_items):
        if key == "stasiun_ui_asrama_01":
            rute_biru_items.insert(i + 1, ("menwa_asrama_01", [-6.353454, 106.831660]))
            break
    
    # Convert back to a dictionary
    route_data["RUTE_BIRU"] = dict(rute_biru_items)

    train['ts'] = pd.to_datetime(train['ts'])
    test['ts'] = pd.to_datetime(test['ts'])
    
    # Build a single graph and store flow coordinates
    G = nx.Graph()
    flow_coords_red = []
    flow_coords_blue = []
    
    # Add flow coordinates to graph
    for track, points in road_data.items():
        for i in range(len(points) - 1):
            coord1 = (points[i]["latitude"], points[i]["longitude"])
            coord2 = (points[i + 1]["latitude"], points[i + 1]["longitude"])
            dist = geodesic(coord1, coord2).meters  # Compute road distance
    
            G.add_edge(coord1, coord2, weight=dist)
            if "RED" in track:
                flow_coords_red.append(coord1)
            else:
                flow_coords_blue.append(coord1)
                
    self_loops = list(nx.selfloop_edges(G))
    G.remove_edges_from(self_loops)
    
    # Convert flow coordinates to radians and build BallTree for fast lookup
    flow_coords_red_radians = np.radians(flow_coords_red)
    flow_coords_blue_radians = np.radians(flow_coords_blue)
    flow_tree_red = BallTree(flow_coords_red_radians, metric="haversine")
    flow_tree_blue = BallTree(flow_coords_blue_radians, metric="haversine")
    
    
    # Functions to find the nearest flow coordinate
    def find_nearest_flow_coord_red(coord):
        coord_radians = np.radians([coord])
        dist, idx = flow_tree_red.query(coord_radians, k=1)
        return tuple(flow_coords_red[idx[0][0]])
    
    def find_nearest_flow_coord_blue(coord):
        coord_radians = np.radians([coord])
        dist, idx = flow_tree_blue.query(coord_radians, k=1)
        return tuple(flow_coords_blue[idx[0][0]])
    
    # Precompute shortest distances to all Red & Blue bus stops
    flow_to_busstop_distances = {}
    stop_segments_red = {}
    stop_segments_blue = {}
    segment_indices_red = {}
    segment_indices_blue = {}
    
    # Get bus stop lists and their sequences
    red_stops = [(name, find_nearest_flow_coord_red(tuple(coord))) for name, coord in route_data["RUTE_MERAH"].items()]
    blue_stops = [(name, find_nearest_flow_coord_blue(tuple(coord))) for name, coord in route_data["RUTE_BIRU"].items()]
    
    # Compute road distances between adjacent bus stops using flow coordinates
    for i in range(len(red_stops) - 1):
        stop1_name, stop1_coord = red_stops[i]
        stop2_name, stop2_coord = red_stops[i + 1]
    
        try:
            segment_distance = nx.shortest_path_length(G, source=stop1_coord, target=stop2_coord, weight="weight")
            stop_segments_red[(stop1_name, stop2_name)] = segment_distance
            segment_indices_red[(stop1_name, stop2_name)] = i + 1
        except nx.NetworkXNoPath:
            stop_segments_red[(stop1_name, stop2_name)] = float("inf")
    
    for i in range(len(blue_stops) - 1):
        stop1_name, stop1_coord = blue_stops[i]
        stop2_name, stop2_coord = blue_stops[i + 1]
    
        try:
            segment_distance = nx.shortest_path_length(G, source=stop1_coord, target=stop2_coord, weight="weight")
            stop_segments_blue[(stop1_name, stop2_name)] = segment_distance
            segment_indices_blue[(stop1_name, stop2_name)] = i + 1
        except nx.NetworkXNoPath:
            stop_segments_blue[(stop1_name, stop2_name)] = float("inf")
    
    # for each flow coord, no matter it is red or blue, it will overlap with the opposite color bus stops. But this is fine
    # because either way we dont access the blue stops result list if we get red coordnitaes bus vice versa
    for flow_coord in flow_coords_red + flow_coords_blue:
        red_distances = {}
        blue_distances = {}
        min_red_dist = float("inf")
        min_blue_dist = float("inf")
        nearest_red_stop = None
        nearest_blue_stop = None
    
        for stop_name, stop_coord in red_stops:
            try:
                path_length = nx.shortest_path_length(G, source=flow_coord, target=stop_coord, weight="weight")
                red_distances[stop_name] = path_length
                if path_length < min_red_dist:
                    min_red_dist = path_length
                    nearest_red_stop = stop_name
            except nx.NetworkXNoPath:
                continue
    
        for stop_name, stop_coord in blue_stops:
            try:
                path_length = nx.shortest_path_length(G, source=flow_coord, target=stop_coord, weight="weight")
                blue_distances[stop_name] = path_length
                if path_length < min_blue_dist:
                    min_blue_dist = path_length
                    nearest_blue_stop = stop_name
            except nx.NetworkXNoPath:
                continue
    
        flow_to_busstop_distances[flow_coord] = {
            "red_bus_stops": red_distances,
            "blue_bus_stops": blue_distances,
            "nearest_red_stop": nearest_red_stop,
            "nearest_red_distance": min_red_dist,
            "nearest_blue_stop": nearest_blue_stop,
            "nearest_blue_distance": min_blue_dist
        }
        # Function to compute halte segment for a given bus observation
    def compute_halte_segment(bus_position, route_color):
        if route_color == "red":
            nearest_road_coord = find_nearest_flow_coord_red(bus_position)
            stop_list = red_stops
            stop_segments = stop_segments_red
            segment_indices = segment_indices_red
        else:
            nearest_road_coord = find_nearest_flow_coord_blue(bus_position)
            stop_list = blue_stops
            stop_segments = stop_segments_blue
            segment_indices = segment_indices_blue
    
        data = flow_to_busstop_distances.get(nearest_road_coord, {})
        nearest_stop = data.get(f"nearest_{route_color}_stop")
    
        stop_idx = [i for i, (name, _) in enumerate(stop_list) if name == nearest_stop][0]
        if route_color == "red":
            neighbor1 = stop_list[stop_idx - 1][0] if stop_idx > 0 else 'menwa_asrama_01'
            neighbor2 = stop_list[stop_idx + 1][0] if stop_idx < len(stop_list) - 1 else None
        else:
            neighbor1 = stop_list[stop_idx - 1][0] if stop_idx > 0 else 'menwa_asrama_01'
            neighbor2 = stop_list[stop_idx + 1][0] if stop_idx < len(stop_list) - 1 else None
    
        bus_to_neighbor1 = data.get(route_color + "_bus_stops", {}).get(neighbor1, float("inf"))
        bus_to_neighbor2 = data.get(route_color + "_bus_stops", {}).get(neighbor2, float("inf"))
    
        if neighbor1 == 'menwa_asrama_01' and stop_idx == 0:
            stop_to_neighbor1 = stop_segments.get((neighbor1, 'asrama_ui_01_end'), float("inf")) + 85
        else:
            stop_to_neighbor1 = stop_segments.get((neighbor1, nearest_stop), float("inf")) #harusnya two way distance
    
        stop_to_neighbor2 = stop_segments.get((nearest_stop, neighbor2), float("inf")) #harusnya two way distance
    
        if bus_to_neighbor1 <= stop_to_neighbor1:
            halte_segment = (neighbor1, nearest_stop)
        else:
            halte_segment = (nearest_stop, neighbor2)
    
        if route_color == "red":
            segment_index = segment_indices.get(halte_segment, 18)
        else:
            segment_index = segment_indices.get(halte_segment, 19)
    
        bus_to_prev = data.get(route_color + "_bus_stops", {}).get(halte_segment[0], float("inf"))
        bus_to_next = data.get(route_color + "_bus_stops", {}).get(halte_segment[1], float("inf"))
        
        return {
            "halte_segment": halte_segment,
            "distPrevStop": bus_to_prev,
            "distNextStop": bus_to_next,
            "segment": segment_index
        }
    
    # Filter hanya data dengan ts >= 06:50
    train = train[train['ts'].dt.time >= pd.to_datetime("06:50").time()]
    
    # Reset index
    train = train.reset_index(drop=True)
    
    # Ambil fitur latitude dan longitude
    gps_data = train[['lat', 'lon']].values
    
    # Terapkan DBSCAN
    eps=0.0002 # hasil eksperimen
    dbscan = DBSCAN(eps=eps, min_samples=5).fit(gps_data)
    train['cluster'] = dbscan.labels_
    
    # Hapus outlier dari train
    train = train[train['cluster'] != -1].drop(columns=['cluster']).reset_index(drop=True)
    
    train[["halte_segment", "distPrevStop", "distNextStop", "segment"]] = train.apply(
        lambda row: pd.Series(compute_halte_segment((row["lat"], row["lon"]), row["color"])), axis=1
    )
    test[["halte_segment", "distPrevStop", "distNextStop", "segment"]] = test.apply(
        lambda row: pd.Series(compute_halte_segment((row["lat"], row["lon"]), row["color"])), axis=1
    )
    
    # Copy-paste your GeoJSON data as a Python dictionary
    geojson_data = {
      "type": "FeatureCollection",
      "features": [
        {
          "type": "Feature",
          "properties": {},
          "geometry": {
            "coordinates": [
              [
                [
                  106.8247764268774,
                  -6.354178669416797
                ],
                [
                  106.82260653511179,
                  -6.361201261495694
                ],
                [
                  106.8233762430446,
                  -6.360793950314687
                ],
                [
                  106.82530743297713,
                  -6.35427268502275
                ],
                [
                  106.8247764268774,
                  -6.354178669416797
                ]
              ]
            ],
            "type": "Polygon"
          }
        }
      ]
    }
    
    # Extract the polygon
    polygon = shape(geojson_data["features"][0]["geometry"])
    train['hutan'] = 0  # Step 1: Default value
    test['hutan'] = 0  # Step 1: Default value
    def mark_hutan(df, polygon):
        df['hutan'] = df.apply(lambda row: 1 if polygon.contains(Point(row['lon'], row['lat'])) else row.get('hutan', 0), axis=1)
        return df
    
    # Apply function to your Bikun dataset
    train = mark_hutan(train, polygon)
    test = mark_hutan(test, polygon)
    
    # Extract time
    train['time'] = train['ts'].dt.time
    test['time'] = test['ts'].dt.time
    
    # Define time range
    start_time = pd.to_datetime("06:50").time()
    end_time = pd.to_datetime("09:30").time()
    
    # Filter rows where hutan == 1 and time is outside the range
    outside_time_hutan = train[(train['hutan'] == 1) & 
                               ((train['time'] < start_time) | (train['time'] > end_time))]
    
    # Exclude rows in outside_time_hutan
    train = train.drop(outside_time_hutan.index).reset_index(drop=True)
    
    def process_hutan_biru(train):
        """
        Processes the dataset to update the 'hutan' column and adjust 'distNextStop' and 'distPrevStop' accordingly.
        """
        # Reference coordinates
        coord1 = (-6.3483937, 106.8297658)
        coord2 = (-6.3489656, 106.8265925)
        
        # Step 2: Process rows sequentially (iterative approach)
        for idx in range(1, len(train)):
            row = train.iloc[idx]
            prev_row = train.iloc[idx - 1]
            
            # Step 3: Check if current row meets the hutan condition (segment 8, color blue, time range)
            if (row['segment'] == 8 and row['color'] == 'blue' and 
                pd.to_datetime("06:50").time() <= row['ts'].time() <= pd.to_datetime("09:30").time()):
                
                # Step 4: Look back for the most recent segment >= 10 within 20 minutes
                for prev_idx in range(idx - 1, -1, -1):  # Iterate backwards
                    prev_row = train.iloc[prev_idx]
                    if abs((row['ts'] - prev_row['ts']).total_seconds()) > 1200:
                        break  # Stop if time difference exceeds 20 minutes
                    if prev_row['segment'] >= 10:
                        if prev_row['segment'] != 19 and prev_row['segment'] != 18:
                            train.at[idx, 'hutan'] = 1
                            train.at[idx, 'distNextStop'] = 2000 - train.at[idx, 'distPrevStop']
                        break  # Stop after finding first segment >= 10
            
            # Step 5: Propagate hutan = 1 if segment is 7 or 19 and previous row had hutan = 1
            elif (row['color'] == 'blue' and row['segment'] in [2, 7, 18, 19] and 
                  pd.to_datetime("06:50").time() <= row['ts'].time() <= pd.to_datetime("09:30").time() and
                  abs((row['ts'] - prev_row['ts']).total_seconds()) <= 600):  # Ensure time difference is below 10 minutes
                
                if prev_row['hutan'] == 1:
                    train.at[idx, 'hutan'] = 1
                    
                    # Compute distances only if this condition applies
                    current_coord = (row['lat'], row['lon'])
                    distance_to_coord1 = geodesic(current_coord, coord1).meters
                    if distance_to_coord1 < 346:
                        train.at[idx, 'distNextStop'] = distance_to_coord1
                    else:
                        distance_to_coord2 = geodesic(current_coord, coord2).meters + 346
                        train.at[idx, 'distNextStop'] = distance_to_coord2
                    
                    train.at[idx, 'distPrevStop'] = 2000 - train.at[idx, 'distNextStop']
    
    process_hutan_biru(train)
    process_hutan_biru(test)
    
    
    def process_hutan_merah(train):
        """
        Processes the dataset to update the 'hutan' column and adjust 'distNextStop' and 'distPrevStop' accordingly.
        """
        # Reference coordinates
        coord1 = (-6.3483937, 106.8297658)
        coord2 = (-6.3489656, 106.8265925)
        
        # Step 2: Process rows sequentially (iterative approach)
        for idx in range(1, len(train)):
            row = train.iloc[idx]
            prev_row = train.iloc[idx - 1]
            
            # Step 3: Check if current row meets the hutan condition (segment 8, color blue, time range)
            if (row['color'] == 'red' and row['hutan'] == 1):
                # Compute distances only if this condition applies
                current_coord = (row['lat'], row['lon'])
                distance_to_coord1 = geodesic(current_coord, coord1).meters
                if distance_to_coord1 < 346:
                    train.at[idx, 'distNextStop'] = distance_to_coord1
                else:
                    distance_to_coord2 = geodesic(current_coord, coord2).meters + 346
                    train.at[idx, 'distNextStop'] = distance_to_coord2
                
                train.at[idx, 'distPrevStop'] = 2000 - train.at[idx, 'distNextStop']
               
            
            # Step 5: Propagate hutan = 1 if segment is 7 or 19 and previous row had hutan = 1
            elif (row['color'] == 'red' and row['segment'] in [2, 12, 17, 18] and 
                  pd.to_datetime("06:50").time() <= row['ts'].time() <= pd.to_datetime("09:30").time() and
                  abs((row['ts'] - prev_row['ts']).total_seconds()) <= 600):  # Ensure time difference is below 10 minutes
                
                if prev_row['hutan'] == 1:
                    train.at[idx, 'hutan'] = 1
                    
                    # Compute distances only if this condition applies
                    current_coord = (row['lat'], row['lon'])
                    distance_to_coord1 = geodesic(current_coord, coord1).meters
                    if distance_to_coord1 < 346:
                        train.at[idx, 'distNextStop'] = distance_to_coord1
                    else:
                        distance_to_coord2 = geodesic(current_coord, coord2).meters + 346
                        train.at[idx, 'distNextStop'] = distance_to_coord2
                    
                    train.at[idx, 'distPrevStop'] = 2000 - train.at[idx, 'distNextStop']
    
    process_hutan_merah(train)
    process_hutan_merah(test)
    
    geojson_data = {
      "type": "FeatureCollection",
      "features": [
        {
          "type": "Feature",
          "properties": {},
          "geometry": {
            "coordinates": [
              [
                [
                  106.8247802947617,
                  -6.354155567531905
                ],
                [
                  106.82530713619576,
                  -6.354256553228552
                ],
                [
                  106.82706555151407,
                  -6.348815068726225
                ],
                [
                  106.8279471433462,
                  -6.348413875486003
                ],
                [
                  106.8289288714754,
                  -6.348491681033693
                ],
                [
                  106.82896261737767,
                  -6.3483194704355626
                ],
                [
                  106.82979949876028,
                  -6.348442569551683
                ],
                [
                  106.8298307201585,
                  -6.348247349791976
                ],
                [
                  106.82789632250012,
                  -6.3478763346527245
                ],
                [
                  106.82658203871176,
                  -6.348353755805292
                ],
                [
                  106.8247802947617,
                  -6.354155567531905
                ]
              ]
            ],
            "type": "Polygon"
          }
        }
      ]
    }
    
    # Extract the polygon
    polygon = shape(geojson_data["features"][0]["geometry"])
    train['last_segment'] = 0  # Step 1: Default value
    test['last_segment'] = 0  # Step 1: Default value
    def mark_last_segment(df, polygon):
        df['last_segment'] = df.apply(
            lambda row: 1 if row['hutan'] == 0 and polygon.contains(Point(row['lon'], row['lat'])) else row.get('last_segment', 0),
            axis=1
        )
        return df
    
    
    # Apply function to your Bikun dataset
    train = mark_last_segment(train, polygon)
    test = mark_last_segment(test, polygon)
    
    def update_segments(train):
        train.loc[train['hutan'] == 1, 'segment'] = 0
        train.loc[(train['last_segment'] == 1) & (train['color'] == 'red'), 'segment'] = 19
        train.loc[(train['last_segment'] == 1) & (train['color'] == 'blue'), 'segment'] = 20
        return train
    
    
    train = update_segments(train)
    test = update_segments(test)
    
    def abnormal_segment_1_18_red(train):
       # Compute segment differences
        train["segment_diff"] = train["segment"].diff()
        
        # Store results
        correction_counts = {"should_be_1": 0, "should_be_18": 0}
        corrected_indices = []
        
        # Iterate through all rows
        for idx in range(1, len(train)):
            if train.loc[idx, "segment"] in [1, 18] and train.loc[idx, "color"] == "red":
                prev_idx = idx - 1
                # Iterate backwards to find the most recent valid previous segment within 5 minutes (300 seconds)
                while prev_idx >= 0 and abs((train.loc[prev_idx, "ts"] - train.loc[idx, "ts"]).total_seconds()) <= 300:
                    prev_segment = train.loc[prev_idx, "segment"]
                    if prev_segment not in [1, 18]:
                        break
                    prev_idx -= 1
                
                if prev_idx >= 0:
                    # Determine correction
                    prev_segment = train.loc[prev_idx, "segment"]
                    if prev_segment in [19]:
                        if train.at[idx, "segment"] == 18 :
                            train.at[idx, "segment"] = 1
                            correction_counts["should_be_1"] += 1
                            corrected_indices.append(idx)
        
                            # Adjust distances
                            segment_key = ('asrama_ui_01', 'menwa_stasiun_ui_01')
                            whole_distance = stop_segments_red[segment_key]
                            train.at[idx, "distNextStop"] = train.at[idx, "distPrevStop"]
                            if train.at[idx, "distNextStop"] > whole_distance:
                                train.at[idx, "distNextStop"] = whole_distance
                            train.at[idx, "distPrevStop"] = whole_distance - train.at[idx, "distNextStop"]
                        
                    else:
                        if train.at[idx, "segment"] == 1 :
                            train.at[idx, "segment"] = 18
                            correction_counts["should_be_18"] += 1
                            corrected_indices.append(idx)
        
                            # Adjust distances
                            segment_key = ('asrama_ui_01', 'menwa_stasiun_ui_01')
                            whole_distance = stop_segments_red[segment_key]
                            train.at[idx, "distPrevStop"] = train.at[idx, "distNextStop"]
                            train.at[idx, "distNextStop"] = whole_distance - train.at[idx, "distPrevStop"]
                    
        
        # Create a DataFrame for corrected indices
        corrected_df = pd.DataFrame({"corrected_index": corrected_indices})
        
        return correction_counts, corrected_df
    
    a,b = abnormal_segment_1_18_red(train)
    a,b = abnormal_segment_1_18_red(test)
    
    def abnormal_segment_1_19_blue(train):
       # Compute segment differences
        train["segment_diff"] = train["segment"].diff()
        
        # Store results
        correction_counts = {"should_be_1": 0, "should_be_19": 0}
        corrected_indices = []
        
        # Iterate through all rows
        for idx in range(1, len(train)):
            if train.loc[idx, "segment"] in [1, 19] and train.loc[idx, "color"] == "blue":
                prev_idx = idx - 1
                # Iterate backwards to find the most recent valid previous segment within 5 minutes (300 seconds)
                while prev_idx >= 0 and abs((train.loc[prev_idx, "ts"] - train.loc[idx, "ts"]).total_seconds()) <= 300:
                    prev_segment = train.loc[prev_idx, "segment"]
                    if prev_segment not in [1, 19]:
                        break
                    prev_idx -= 1
                
                if prev_idx >= 0:
                    # Determine correction
                    prev_segment = train.loc[prev_idx, "segment"]
                    if prev_segment in [20]:
                        if train.at[idx, "segment"] == 19 :
                            train.at[idx, "segment"] = 1
                            correction_counts["should_be_1"] += 1
                            corrected_indices.append(idx)
        
                            # Adjust distances
                            segment_key = ('asrama_ui_01', 'menwa_stasiun_ui_01')
                            whole_distance = stop_segments_blue[segment_key]
                            train.at[idx, "distNextStop"] = train.at[idx, "distPrevStop"]
                            if train.at[idx, "distNextStop"] > whole_distance:
                                train.at[idx, "distNextStop"] = whole_distance
                            train.at[idx, "distPrevStop"] = whole_distance - train.at[idx, "distNextStop"]
                        
                    else:
                        if train.at[idx, "segment"] == 1 :
                            train.at[idx, "segment"] = 19
                            correction_counts["should_be_19"] += 1
                            corrected_indices.append(idx)
        
                            # Adjust distances
                            segment_key = ('asrama_ui_01', 'menwa_stasiun_ui_01')
                            whole_distance = stop_segments_blue[segment_key]
                            train.at[idx, "distPrevStop"] = train.at[idx, "distNextStop"]
                            train.at[idx, "distNextStop"] = whole_distance - train.at[idx, "distPrevStop"]
                    
        
        # Create a DataFrame for corrected indices
        corrected_df = pd.DataFrame({"corrected_index": corrected_indices})
        
        return correction_counts, corrected_df
    
    a,b = abnormal_segment_1_19_blue(train)
    a,b = abnormal_segment_1_19_blue(test)
    
    def abnormal_segment_2_17_red(train):
       # Compute segment differences
        train["segment_diff"] = train["segment"].diff()
        
        # Store results
        correction_counts = {"should_be_2": 0, "should_be_17": 0}
        corrected_indices = []
        
        # Iterate through all rows
        for idx in range(1, len(train)):
            if train.loc[idx, "segment"] in [2, 17] and train.loc[idx, "color"] == "red":
                prev_idx = idx - 1
                # Iterate backwards to find the most recent valid previous segment within 5 minutes (300 seconds)
                while prev_idx >= 0 and abs((train.loc[prev_idx, "ts"] - train.loc[idx, "ts"]).total_seconds()) <= 300:
                    prev_segment = train.loc[prev_idx, "segment"]
                    if prev_segment not in [2, 17]:
                        break
                    prev_idx -= 1
                
                if prev_idx >= 0:
                    # Determine correction
                    prev_segment = train.loc[prev_idx, "segment"]
                    if prev_segment in [18,19,1]:
                        if train.at[idx, "segment"] == 17 :
                            train.at[idx, "segment"] = 2
                            correction_counts["should_be_2"] += 1
                            corrected_indices.append(idx)
        
                            # Adjust distances
                            segment_key = ('menwa_stasiun_ui_01', 'stasiun_ui_pfh_01')
                            whole_distance = stop_segments_red[segment_key]
                            train.at[idx, "distPrevStop"] = train.at[idx, "distNextStop"]
                            train.at[idx, "distNextStop"] = whole_distance - train.at[idx, "distPrevStop"]
                        
                    else:
                        if train.at[idx, "segment"] == 2 :
                            train.at[idx, "segment"] = 17
                            correction_counts["should_be_17"] += 1
                            corrected_indices.append(idx)
        
                            # Adjust distances
                            segment_key = ('stasiun_ui_asrama_01', 'menwa_asrama_01')
                            whole_distance = stop_segments_red[segment_key]
                            train.at[idx, "distNextStop"] = train.at[idx, "distPrevStop"]
                            train.at[idx, "distPrevStop"] = whole_distance - train.at[idx, "distNextStop"]
        
        # Create a DataFrame for corrected indices
        corrected_df = pd.DataFrame({"corrected_index": corrected_indices})
        
        return correction_counts, corrected_df
    
    a,b = abnormal_segment_2_17_red(train)
    a,b = abnormal_segment_2_17_red(test)
    
    def abnormal_segment_2_18_blue(train):
       # Compute segment differences
        train["segment_diff"] = train["segment"].diff()
        
        # Store results
        correction_counts = {"should_be_2": 0, "should_be_18": 0}
        corrected_indices = []
        
        # Iterate through all rows
        for idx in range(1, len(train)):
            if train.loc[idx, "segment"] in [2, 18] and train.loc[idx, "color"] == "blue":
                prev_idx = idx - 1
                # Iterate backwards to find the most recent valid previous segment within 5 minutes (300 seconds)
                while prev_idx >= 0 and abs((train.loc[prev_idx, "ts"] - train.loc[idx, "ts"]).total_seconds()) <= 300:
                    prev_segment = train.loc[prev_idx, "segment"]
                    if prev_segment not in [2, 18]:
                        break
                    prev_idx -= 1
                
                if prev_idx >= 0:
                    # Determine correction
                    prev_segment = train.loc[prev_idx, "segment"]
                    if prev_segment in [19,20,1] :
                        if train.at[idx, "segment"] == 18 :
                            train.at[idx, "segment"] = 2
                            correction_counts["should_be_2"] += 1
                            corrected_indices.append(idx)
        
                            # Adjust distances
                            segment_key = ('menwa_stasiun_ui_01', 'stasiun_ui_pfh_01')
                            whole_distance = stop_segments_blue[segment_key]
                            train.at[idx, "distPrevStop"] = train.at[idx, "distNextStop"]
                            train.at[idx, "distNextStop"] = whole_distance - train.at[idx, "distPrevStop"]
                        
                    else:
                        if train.at[idx, "segment"] == 2 :
                            train.at[idx, "segment"] = 18
                            correction_counts["should_be_18"] += 1
                            corrected_indices.append(idx)
        
                            # Adjust distances
                            segment_key = ('stasiun_ui_asrama_01', 'menwa_asrama_01')
                            whole_distance = stop_segments_blue[segment_key]
                            train.at[idx, "distNextStop"] = train.at[idx, "distPrevStop"]
                            train.at[idx, "distPrevStop"] = whole_distance - train.at[idx, "distNextStop"]
        
        # Create a DataFrame for corrected indices
        corrected_df = pd.DataFrame({"corrected_index": corrected_indices})
        
        return correction_counts, corrected_df
    
    a,b = abnormal_segment_2_18_blue(train)
    a,b = abnormal_segment_2_18_blue(test)
    
    def abnormal_segment_3_16_red(train):
       # Compute segment differences
        train["segment_diff"] = train["segment"].diff()
        
        # Store results
        correction_counts = {"should_be_3": 0, "should_be_16": 0}
        corrected_indices = []
        
        # Iterate through all rows
        for idx in range(1, len(train)):
            if train.loc[idx, "segment"] in [3, 16] and train.loc[idx, "color"] == "red":
                prev_idx = idx - 1
                # Iterate backwards to find the most recent valid previous segment within 5 minutes (300 seconds)
                while prev_idx >= 0 and abs((train.loc[prev_idx, "ts"] - train.loc[idx, "ts"]).total_seconds()) <= 300:
                    prev_segment = train.loc[prev_idx, "segment"]
                    if prev_segment not in [3, 16]:
                        break
                    prev_idx -= 1
                
                if prev_idx >= 0:
                    # Determine correction
                    prev_segment = train.loc[prev_idx, "segment"]
                    if prev_segment in [17, 18, 19, 1, 2] :
                        if train.at[idx, "segment"] == 16 :
                            train.at[idx, "segment"] = 3
                            correction_counts["should_be_3"] += 1
                            corrected_indices.append(idx)
        
                            # Adjust distances
                            segment_key = ('stasiun_ui_pfh_01', 'fh_1')
                            whole_distance = stop_segments_red[segment_key]
                            train.at[idx, "distPrevStop"] = train.at[idx, "distNextStop"]
                            train.at[idx, "distNextStop"] = whole_distance - train.at[idx, "distPrevStop"]
                        
                    else:
                        if train.at[idx, "segment"] == 3 :
                            train.at[idx, "segment"] = 16
                            correction_counts["should_be_16"] += 1
                            corrected_indices.append(idx)
        
                            # Adjust distances
                            segment_key = ('fpsi_1', 'stasiun_ui_asrama_01')
                            whole_distance = stop_segments_red[segment_key]
                            train.at[idx, "distNextStop"] = train.at[idx, "distPrevStop"]
                            train.at[idx, "distPrevStop"] = whole_distance - train.at[idx, "distNextStop"]
        
        # Create a DataFrame for corrected indices
        corrected_df = pd.DataFrame({"corrected_index": corrected_indices})
        
        return correction_counts, corrected_df
    
    a,b = abnormal_segment_3_16_red(train)
    a,b = abnormal_segment_3_16_red(test)
    
    def abnormal_segment_3_17_blue(train):
       # Compute segment differences
        train["segment_diff"] = train["segment"].diff()
        
        # Store results
        correction_counts = {"should_be_3": 0, "should_be_17": 0}
        corrected_indices = []
        
        # Iterate through all rows
        for idx in range(1, len(train)):
            if (
                train.loc[idx, "segment"] in [3, 17]
                and train.loc[idx, "color"] == "blue"
                and not (train.loc[idx, "ts"].time() >= datetime.time(6, 50) and train.loc[idx, "ts"].time() <= datetime.time(9, 30))
            ):
                prev_idx = idx - 1
                # Iterate backwards to find the most recent valid previous segment within 5 minutes (300 seconds)
                while prev_idx >= 0 and abs((train.loc[prev_idx, "ts"] - train.loc[idx, "ts"]).total_seconds()) <= 300:
                    prev_segment = train.loc[prev_idx, "segment"]
                    if prev_segment not in [3, 17]:
                        break
                    prev_idx -= 1
                
                if prev_idx >= 0:
                    # Determine correction
                    prev_segment = train.loc[prev_idx, "segment"]
                    if prev_segment in [18,19,20,1, 2] :
                        if train.at[idx, "segment"] == 17 :
                            train.at[idx, "segment"] = 3
                            correction_counts["should_be_3"] += 1
                            corrected_indices.append(idx)
        
                            # Adjust distances
                            segment_key = ('stasiun_ui_pfh_01', 'fpsi_0')
                            whole_distance = stop_segments_blue[segment_key]
                            train.at[idx, "distPrevStop"] = train.at[idx, "distNextStop"]
                            train.at[idx, "distNextStop"] = whole_distance - train.at[idx, "distPrevStop"]
                        
                    else:
                        if train.at[idx, "segment"] == 3 :
                            train.at[idx, "segment"] = 17
                            correction_counts["should_be_17"] += 1
                            corrected_indices.append(idx)
        
                            # Adjust distances
                            segment_key = ('fh_0', 'stasiun_ui_asrama_01')
                            whole_distance = stop_segments_blue[segment_key]
                            train.at[idx, "distNextStop"] = train.at[idx, "distPrevStop"]
                            train.at[idx, "distPrevStop"] = whole_distance - train.at[idx, "distNextStop"]
        
        # Create a DataFrame for corrected indices
        corrected_df = pd.DataFrame({"corrected_index": corrected_indices})
        
        return correction_counts, corrected_df
    
    a,b = abnormal_segment_3_17_blue(train)
    a,b = abnormal_segment_3_17_blue(test)
    
    def update_distNextStop(train):
        condition = ((train['segment'] == 18) & (train['color'] == 'red')) | ((train['segment'] == 19) & (train['color'] == 'blue'))
        
        # Compute new distNextStop
        train.loc[condition, 'distNextStop'] = 756 - train.loc[condition, 'distPrevStop']
        
        # Ensure no negative values
        train.loc[train['distNextStop'] < 0, 'distNextStop'] = 0
    
        return train
    
    
    train = update_distNextStop(train)
    test = update_distNextStop(test)
    # Example function to process the dataframe
    def update_segments(df, stop_segments_red, segment_indices_red, stop_segments_blue, segment_indices_blue):
        def update_row(row):
            if row["distPrevStop"] == 0:
                color = row["color"]
                
                # Select the correct mappings based on color
                stop_segments = stop_segments_red if color == "red" else stop_segments_blue
                segment_indices = segment_indices_red if color == "red" else segment_indices_blue
    
                # Decrement segment, wrap around if it becomes 0
                new_segment = row["segment"] - 1
                if new_segment == 0:
                    new_segment = 18  # Wrap around to 18
                
                # Update values
                row["segment"] = new_segment
                row["distPrevStop"] = list(stop_segments.values())[int(new_segment) - 1]  # Get distance by index
                row["distNextStop"] = 0
                row["halte_segment"] = list(segment_indices.keys())[int(new_segment) - 1]  # Get halte_segment by index
            
            return row
    
        return df.apply(update_row, axis=1)
    
    # Assuming you have a DataFrame named `train`
    train = update_segments(train, stop_segments_red, segment_indices_red, stop_segments_blue, segment_indices_blue)
    test = update_segments(test, stop_segments_red, segment_indices_red, stop_segments_blue, segment_indices_blue)
    
    train = train.drop(index=range(9156, 11138)).reset_index(drop=True)
    
    def compute_rta(df):
        rta_list = [None] * len(df)  # Initialize with None
    
        for i in range(len(df)):
            current_segment = df.loc[i, "segment"]
            current_ts = df.loc[i, "ts"]
            current_color = df.loc[i, "color"]
    
            # Determine the max segment based on color
            max_segment = 20 if current_color == "blue" else 19
            next_segment = 1 if current_segment == max_segment else current_segment + 1
    
            # Find the first occurrence where the segment changes
            segment_change_idx = df[(df.index > i) & (df["segment"] != current_segment) & (df["color"] == current_color)].index
            if segment_change_idx.empty:
                rta_list[i] = np.nan
                continue
    
            first_change_idx = segment_change_idx[0]
            next_rows = df[(df.index >= first_change_idx) & (df["segment"] == next_segment) & (df["color"] == current_color)]
            
            # Ensure the next segment is found immediately after the first segment change
            if not next_rows.empty and (next_rows.index[0] == first_change_idx):
                first_next_segment_ts = next_rows.iloc[0]["ts"]
                rta_list[i] = (first_next_segment_ts - current_ts).total_seconds()
            else:
                rta_list[i] = np.nan
    
        df["rta"] = rta_list
        return df
        
    train = compute_rta(train)
    
    # Identify rows where IMEI changes
    imei_change_indices = train["imei"] != train["imei"].shift(-1)
    
    # Iterate over those rows and set `rta` to NaN for all previous rows in the same segment
    for idx in train[imei_change_indices].index:
        segment = train.loc[idx, "segment"]  # Get the segment
        imei = train.loc[idx, "imei"]  # Get the imei
        
        # Set `rta` to NaN for all previous rows with the same `imei` and `segment`
        train.loc[
            (train["imei"] == imei) & 
            (train["segment"] == segment) & 
            (train.index <= idx), 
            "rta"
        ] = np.nan
        
    # Impute NaN values in 'rta' with its median
    train['rta'].fillna(train['rta'].median(), inplace=True)

    train["color"] = train["color"].astype("category")
    train["segment"] = train["segment"].astype("category")
    test["color"] = test["color"].astype("category")
    test["segment"] = test["segment"].astype("category")
    
    return train, test


In [142]:
def infer(models, train_processed, test_processed):
   
    
    
    # List of columns to drop
    cols_to_drop = ["lat", "lon", "ts", "Id", "halte_segment", "rta", "distPrevStop", "segment_diff", "last_segment", "time", "hutan"]
    
    # Drop only existing columns from test set
    test_processed = test_processed.drop(columns=[col for col in cols_to_drop if col in test_processed.columns])
    
    # Define features and target
    X_train = train_processed.drop(columns=["lat", "lon", "ts", "imei", "halte_segment", "rta", "distPrevStop", "segment_diff", "last_segment", "time", "hutan"], errors='ignore')
    y_train = train_processed["rta"]

    X_train["color"] = X_train["color"].cat.codes
    X_train["segment"] = X_train["segment"].cat.codes
    test_processed["color"] = test_processed["color"].cat.codes
    test_processed["segment"] = test_processed["segment"].cat.codes
    
    # Train all models
    trained_models = []
    for model in models:
        model.fit(X_train, y_train)
        trained_models.append(model)
    
    # Make predictions using an ensemble (average of all models)
    predictions = np.mean([model.predict(test_processed) for model in trained_models], axis=0)
    
    # Clip values to be within [25, 300] seconds
    predictions = np.clip(predictions, a_min=1, a_max=1894)
    
    # Save predictions
    submission = pd.DataFrame({"Id": test_processed.index, "rta": predictions})
    submission.to_csv("submission.csv", index=False)
    
    return submission

SEED = 42  # Set random seed
LABEL = "rta"
# Initialize and train the model beforehand
xgb_model = xgb.XGBRegressor(
    random_state = SEED,
    enable_categorical=True  # Enable categorical support
)
# List of trained models
models = [xgb_model]

In [158]:
train_processed, test_processed = process(train, test)
predictions = infer(models, train_processed, test_processed) # models adalah list yang berisi model-model yang digunakan, yang telah di-fit dengan data train_processed